### Importation des bibliothèques

### Installation des d?pendances (Kaggle-safe)


In [ ]:
import sys
import subprocess
import importlib.util


def ensure_package(import_name: str, pip_spec: str):
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {pip_spec}")
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            pip_spec,
        ])


# Kaggle-safe: do not force-upgrade torch/cuda.
ensure_package("numpy", "numpy")
ensure_package("scipy", "scipy")
ensure_package("pandas", "pandas")
ensure_package("sklearn", "scikit-learn")
ensure_package("yaml", "pyyaml")
ensure_package("gensim", "gensim")
ensure_package("tqdm", "tqdm")
ensure_package("torch", "torch")

print("Dependencies check done")


In [ ]:
import os
import scipy.io
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy.sparse as sp
from tqdm.auto import tqdm
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import normalized_mutual_info_score
import time
import random
import os
from torch.amp import GradScaler, autocast
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary

### Configuration de l'environnement

In [ ]:
from pathlib import Path


def detect_project_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents, cwd / 'Projet_PPD', cwd / 'Projet_PPD' / 'Projet_PPD']
    ordered = []
    seen = set()
    for candidate in candidates:
        try:
            resolved = candidate.resolve()
        except Exception:
            continue
        key = str(resolved)
        if key not in seen:
            seen.add(key)
            ordered.append(resolved)
    for candidate in ordered:
        if (candidate / 'data').exists():
            return candidate
    raise FileNotFoundError('Impossible de trouver le dossier projet contenant data/ et output/.')


PROJECT_ROOT = detect_project_root()
DATA_ROOT = str(PROJECT_ROOT / 'data')
SAVE_ROOT_NSTM = str(PROJECT_ROOT / 'output' / 'NSTM')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"L'entrainement se fera sur : {DEVICE}")

DATASETS = ['20NG', 'IMDB', 'YahooAnswer', 'AGNews']
K_VALUES = [50, 100]
SEED = 42

if not os.path.exists(SAVE_ROOT_NSTM):
    os.makedirs(SAVE_ROOT_NSTM)


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f'Graine fixee : {seed}')


set_seed(SEED)


### Chargement des données et fonctions métriques d'évaluation

In [ ]:
def load_dataset_(name):
    path = os.path.join(DATA_ROOT, name)
    train = sp.load_npz(os.path.join(path, "train_bow.npz")).toarray()
    test = sp.load_npz(os.path.join(path, "test_bow.npz")).toarray()
    vocab = [w.strip() for w in open(os.path.join(path, "vocab.txt"), encoding="utf-8")]
    test_labels = np.loadtxt(os.path.join(path, "test_labels.txt"), dtype=int)
    return train, test, vocab, test_labels

def get_metrics_fast(beta, theta, y_test, vocab):
    # Topic Diversity (TD)
    num_topics = beta.shape[0]
    list_w = []
    for k in range(num_topics):
        idx = beta[k].argsort()[-10:][::-1]
        list_w.extend([vocab[i] for i in idx])
    td = len(set(list_w)) / len(list_w)
    
    # Clustering (NMI)
    preds = theta.argmax(1)
    nmi = normalized_mutual_info_score(y_test, preds)
    
    # Purity
    y_voted = np.zeros(y_test.shape)
    labels = np.unique(y_test)
    for cluster in np.unique(preds):
        mask = (preds == cluster)
        if mask.any():
            hist, _ = np.histogram(y_test[mask], bins=np.append(labels, labels.max() + 1))
            y_voted[mask] = labels[np.argmax(hist)]
    purity = np.mean(y_voted == y_test)
    
    return td, nmi, purity

### Implémentation du modèle NSTM

In [ ]:
class NSTM_Model(nn.Module):
    def __init__(self, vocab_size, num_topics, embed_size=300):
        super(NSTM_Model, self).__init__()
        # Initialisation Xavier pour la stabilité
        self.rho = nn.Parameter(torch.empty(vocab_size, embed_size))
        self.alpha = nn.Parameter(torch.empty(num_topics, embed_size))
        nn.init.xavier_uniform_(self.rho)
        nn.init.xavier_uniform_(self.alpha)
        
        self.encoder = nn.Sequential(
            nn.Linear(vocab_size, 500),
            nn.ReLU(),
            nn.BatchNorm1d(500),
            nn.Dropout(0.2),
            nn.Linear(500, num_topics),
            nn.Softmax(dim=-1)
        )

    def get_beta(self, temp=0.2):
        dist = torch.mm(F.normalize(self.alpha), F.normalize(self.rho).t())
        return F.softmax(dist / temp, dim=-1)

    def forward(self, x):
        theta = self.encoder(x)
        beta = self.get_beta()
        recon = torch.mm(theta, beta)
        return recon, theta

### Boucle d'entraînement du modèle NSTM

In [ ]:
def sinkhorn_loss(x, y, M, eps=0.1, n_iter=20):
    M = M.to(x.device)
    K = torch.exp(-torch.clamp(M / eps, max=50)) 
    u = torch.ones_like(x) / (x.shape[1] + 1e-12)

    for _ in range(n_iter):
        v = y / (torch.matmul(u, K) + 1e-12)
        u = x / (torch.matmul(v, K.t()) + 1e-12)

    if torch.isnan(u).any() or torch.isnan(v).any():
        return torch.tensor(0.0, device=x.device, requires_grad=True)

    cost_matrix_weighted = K * M
    loss = torch.sum(u * torch.matmul(v, cost_matrix_weighted.t()))
    return loss / x.shape[0]



def train_nstm(dataset_name, K, epochs=120, warmup_epochs=40):
    train_bow, _, vocab, _ = load_dataset_(dataset_name)
    model = NSTM_Model(len(vocab), K).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0005) 
    scaler = GradScaler(enabled=(DEVICE.type == 'cuda'))
    loader = DataLoader(TensorDataset(torch.from_numpy(train_bow).float()), 
                        batch_size=64, shuffle=True)

    pbar = tqdm(range(epochs), desc=f"Training {dataset_name} K={K}")
    for epoch in pbar:
        model.train()
        with torch.no_grad():
            rho_norm = F.normalize(model.rho, p=2, dim=1)
            M = 1 - torch.mm(rho_norm, rho_norm.t())

        total_loss = 0
        for batch in loader:
            x = batch[0].to(DEVICE)
            optimizer.zero_grad()

            with autocast(device_type=DEVICE.type):
                recon, _ = model(x)
                x_norm = x / (x.sum(1, keepdim=True) + 1e-10)
                if epoch < warmup_epochs:
                    loss = F.mse_loss(recon, x_norm) * 200
                else:
                    loss = sinkhorn_loss(x_norm, recon, M)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()

        pbar.set_postfix({"loss": f"{total_loss/len(loader):.4f}"})

    return model, vocab

### Évaluation des performances et calcul des métriques

In [ ]:
# --- 1. CONFIGURATION GLOBALE ---
DATASETS = ["20NG", "IMDB", "YahooAnswer", "AGNews"]
K_VALUES = [50, 100]
SAVE_FILE = os.path.join(SAVE_ROOT_NSTM, "NSTM_output.csv")
TIMING_FILE = os.path.join(SAVE_ROOT_NSTM, "NSTM_training_times.csv")
set_seed(42)

results = []
if os.path.exists(SAVE_FILE):
    results = pd.read_csv(SAVE_FILE).to_dict('records')
    print(f" {len(results)} r?sultats d?j? trouv?s. Reprise du calcul...")

timing_rows = pd.read_csv(TIMING_FILE).to_dict('records') if os.path.exists(TIMING_FILE) else []

for dataset_name in DATASETS:
    print(f"\n" + "="*60 + f"\n TRAITEMENT DU JEU DE DONN?ES : {dataset_name}\n" + "="*60)

    train_bow, test_bow, vocab, y_test = load_dataset_(dataset_name)

    for K in K_VALUES:
        if any(r['Dataset'] == dataset_name and r['K'] == K for r in results):
            print(f" NSTM  | K={K} | D?j? calcul?.")
            continue

        print(f" NSTM  | K={K} | Entra?nement...")
        total_start = time.perf_counter()

        train_start = time.perf_counter()
        model, vocab = train_nstm(dataset_name, K, epochs=120, warmup_epochs=30)
        train_seconds = time.perf_counter() - train_start
        model.eval()

        with torch.no_grad():
            beta = model.get_beta(temp=0.2).cpu().numpy()
            test_tensor = torch.from_numpy(test_bow).float().to(DEVICE)
            _, theta = model(test_tensor)
            theta = theta.cpu().numpy()

        td, nmi, pur = get_metrics_fast(beta, theta, y_test, vocab)
        total_seconds = time.perf_counter() - total_start

        res = {"Dataset": dataset_name, "K": K, "TD": round(td, 3), "Purity": round(pur, 3), "NMI": round(nmi, 3)}
        results.append(res)
        pd.DataFrame(results).to_csv(SAVE_FILE, index=False)

        trow = {
            'model': 'NSTM', 'dataset': dataset_name, 'K': int(K), 'seed': 42, 'device': str(DEVICE),
            'phase': 'train_metrics', 'train_seconds': float(train_seconds), 'total_seconds': float(total_seconds),
            'train_minutes': float(train_seconds / 60.0), 'total_minutes': float(total_seconds / 60.0),
        }
        timing_rows.append(trow)
        pd.DataFrame(timing_rows).to_csv(TIMING_FILE, index=False)

        print(f"K={K} Termin? | TD: {res['TD']} | Pur: {res['Purity']} | NMI: {res['NMI']} | Time={total_seconds:.2f}s")


df_final = pd.DataFrame(results)
print("\n" + "#"*70 + "\n         Tableau r?cap des r?sultats \n" + "#"*70)
print(df_final.to_string(index=False))

if timing_rows:
    df_t = pd.DataFrame(timing_rows).sort_values(['dataset', 'K']).reset_index(drop=True)
    print("\n--- TIMING NSTM ---")
    print(df_t[['dataset', 'K', 'device', 'train_seconds', 'total_seconds']].to_string(index=False))
    print(f"Saved training times: {TIMING_FILE}")


### Calcul de la Cohérence Sémantique

In [ ]:
def evaluate_coherence_direct(beta, vocab_list, dataset_name, top_n=15):
    # 1. Extraction des mots-clés (Top 15)
    topics_words = []
    for b in beta:
        top_idx = b.argsort()[-top_n:][::-1]
        topics_words.append([vocab_list[i] for i in top_idx])

    # 2. Préparation des textes pour Gensim
    train_bow, _, _, _ = load_dataset_(dataset_name)
    texts = []
    for i in range(train_bow.shape[0]):
        row = train_bow[i]
        idx = row.nonzero()[1] if sp.issparse(row) else row.nonzero()[0]
        if len(idx) > 1:
            texts.append([vocab_list[j] for j in idx])
    
    # 3. Création du dictionnaire Gensim
    dictionary = Dictionary(texts)
    
    # 4. Calcul de la cohérence C_V
    cm = CoherenceModel(
        topics=topics_words, 
        texts=texts, 
        dictionary=dictionary, 
        coherence='c_v', 
        processes=1
    )
    
    return cm.get_coherence()

In [ ]:
# --- CONFIGURATION FINALE ---
DATASETS = ["20NG", "IMDB", "YahooAnswer", "AGNews"]
K_VALUES = [50, 100]
TOPN = 15
SAVE_FILE = "NSTM_Final_Results.csv"
TIMING_FILE_CV = os.path.join(SAVE_ROOT_NSTM, "NSTM_training_times_cv.csv")
set_seed(42)

if os.path.exists(SAVE_FILE):
    df_exist = pd.read_csv(SAVE_FILE)
    nstm_final_results = df_exist.to_dict('records')
    print(f"Fichier trouve : {len(nstm_final_results)} resultats charges.")
else:
    nstm_final_results = []
    print("Aucun resultat existant, demarrage d'une nouvelle etude.")

timing_rows_cv = pd.read_csv(TIMING_FILE_CV).to_dict('records') if os.path.exists(TIMING_FILE_CV) else []

for ds_name in DATASETS:
    for k in K_VALUES:
        deja_fait = any(r['Dataset'] == ds_name and r['K'] == k for r in nstm_final_results)
        if deja_fait:
            print(f"Saut de l'etape : {ds_name} | K={k} (deja present dans le CSV)")
            continue

        print("\n" + "#"*50)
        print(f"TRAITEMENT : {ds_name} | K={k}")
        print("#"*50)

        try:
            total_start = time.perf_counter()
            train_start = time.perf_counter()
            model, vocab = train_nstm(ds_name, k)
            train_seconds = time.perf_counter() - train_start

            with torch.no_grad():
                beta = model.get_beta().cpu().numpy()

            del model
            gc.collect()
            torch.cuda.empty_cache()

            print(f"Calcul de la coherence C_V (Top {TOPN})")
            score_cv = evaluate_coherence_direct(beta, vocab, ds_name, top_n=TOPN)
            total_seconds = time.perf_counter() - total_start

            nstm_final_results.append({"Dataset": ds_name, "K": k, "C_V": round(score_cv, 4)})
            pd.DataFrame(nstm_final_results).to_csv(SAVE_FILE, index=False)

            timing_rows_cv.append({
                'model': 'NSTM', 'dataset': ds_name, 'K': int(k), 'seed': 42, 'device': str(DEVICE),
                'phase': 'train_cv', 'train_seconds': float(train_seconds), 'total_seconds': float(total_seconds),
                'train_minutes': float(train_seconds / 60.0), 'total_minutes': float(total_seconds / 60.0),
            })
            pd.DataFrame(timing_rows_cv).to_csv(TIMING_FILE_CV, index=False)

            print(f"Succes : {ds_name} (K={k}) -> C_V = {score_cv:.4f} | Time={total_seconds:.2f}s")

        except Exception as e:
            print(f"Erreur sur {ds_name} (K={k}) : {e}")
            torch.cuda.empty_cache()
            gc.collect()

print("\n" + "="*50)
print("SYNTHESE DES RESULTATS NSTM")
print("="*50)
df_res = pd.DataFrame(nstm_final_results)
if not df_res.empty:
    pivot_df = df_res.pivot(index="Dataset", columns="K", values="C_V")
    print(pivot_df)

if timing_rows_cv:
    df_t_cv = pd.DataFrame(timing_rows_cv).sort_values(['dataset', 'K']).reset_index(drop=True)
    print("\n--- TIMING NSTM (CV phase) ---")
    print(df_t_cv[['dataset', 'K', 'device', 'train_seconds', 'total_seconds']].to_string(index=False))
    print(f"Saved training times: {TIMING_FILE_CV}")


### Analyse comparative : Résultats originaux vs Reproduction

In [ ]:
# 1. Données du papier
data_papier = {
    'Dataset': ['20NG', '20NG', 'AGNews', 'AGNews', 'IMDB', 'IMDB', 'Yahoo', 'Yahoo'],
    'K': [50, 100, 50, 100, 50, 100, 50, 100],
    'CV_P': [0.395, 0.391, 0.411, 0.421, 0.334, 0.340, 0.390, 0.387],
    'TD_P': [0.427, 0.473, 0.873, 0.832, 0.175, 0.255, 0.658, 0.659],
    'Purity_P': [0.354, 0.383, 0.719, 0.764, 0.658, 0.659, 0.395, 0.405]
}

# 2. Données de la reproduction
data_repro = {
    'Dataset': ['20NG', '20NG', 'AGNews', 'AGNews', 'IMDB', 'IMDB', 'Yahoo', 'Yahoo'],
    'K': [50, 100, 50, 100, 50, 100, 50, 100],
    'CV_R': [0.437, 0.352, 0.483, 0.406, 0.309, 0.285, 0.475, 0.518],
    'TD_R': [0.394, 0.158, 0.618, 0.397, 0.348, 0.204, 0.270, 0.172],
    'Purity_R': [0.268, 0.248, 0.590, 0.626, 0.595, 0.605, 0.236, 0.283]
}

df_p = pd.DataFrame(data_papier)
df_r = pd.DataFrame(data_repro)

# Fusion des tableaux
df_comp = pd.merge(df_p, df_r, on=['Dataset', 'K'])

# Calcul des écarts (Diff)
df_comp['Diff_CV'] = df_comp['CV_R'] - df_comp['CV_P']
df_comp['Diff_TD'] = df_comp['TD_R'] - df_comp['TD_P']
df_comp['Diff_Purity'] = df_comp['Purity_R'] - df_comp['Purity_P']

# Sélection et formatage des colonnes
cols_final = [
    'Dataset', 'K', 
    'CV_P', 'CV_R', 'Diff_CV', 
    'TD_P', 'TD_R', 'Diff_TD', 
    'Purity_P', 'Purity_R', 'Diff_Purity'
]

# Affichage avec arrondi à 3 chiffres
display(df_comp[cols_final].round(3))